In [1]:
import warnings
from pathlib import Path

from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter('ignore', InterpolationWarning)

from input import input
import config
from model import generics, single_ml_model_exp, grid_search_exp
from model.feature_selection import TimeSeriesFeatureSelector
from metrics import metrics
import numpy as np

from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
import pandas as pd

%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\pydoc_data\topics.py' for module 'pydoc_data.topics': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  Fil

In [2]:
# === CELULA DE CONFIGURACAO -- Tarefa 5 (fluxo notebook-only, mesmo padrao
# ja usado pelos notebooks de FS de ARIMA-MLP desde a Tarefa 3.2) ===
# Edite aqui para uma nova rodada: series, experiment_id, grid. Ver
# RUNBOOK.md Secao 1 (series/lag_size) e Secao 3 (convencao de experiment_id).
#
# Tarefa 5 do PLANO_ARQUITETURA.md: generalizacao do arsenal de Feature
# Selection (ja validado em ARIMA-MLP via Additive) para MLP single, via
# SKlearnModel -- confirmado por pre-check (Tarefa 5) que SKlearnModel e
# agnostico a identidade de `model` na mesma forma que Additive ja era
# (fit_predict_ml_schemma so chama model.fit/model.predict), entao o mesmo
# Pipeline([selector, estimator]) funciona sem NENHUMA mudanca em
# single_ml_model_exp.py/grid_search_exp.py. Diferente do hibrido, nao ha
# residuo do ARIMA nem copia de modelo linear pre-treinado -- SKlearnModel
# opera direto sobre a serie bruta (uma celula a menos que os notebooks de
# ARIMA-MLP: sem a celula de "copia do ARIMA").
#
# strategy fixa nesta execucao ('mutual_info') -- k varia como grid (selector__k abaixo), mesmo padrao do hibrido.
model = Pipeline([
    ('selector', TimeSeriesFeatureSelector(strategy='mutual_info')),
    ('estimator', MLPRegressor(activation='logistic', solver='lbfgs')),
])

# Series a rodar nesta execucao (FS_DEV_SERIES por padrao -- tests/model/conftest.py).
# lag_size='auto' medido na Tarefa 5 (pre-check): resolve para os MESMOS
# valores que ja foram medidos para o hibrido ARIMA-MLP (airlines=20,
# austres=1, coloradoRiver=16, sunspot=9) -- confirmado com dado real, nao
# assumido (ver tests/model/test_single_ml_model_exp.py). df_train tem mais
# linhas que o hibrido (ex. airlines: 96 aqui vs. 80 no hibrido) porque
# SKlearnModel janela a serie bruta uma unica vez, sem o janelamento duplo
# implicito do residuo do ARIMA.
fs_series_list = ['airlines.txt', 'austres.txt', 'coloradoRiver.txt', 'sunspot.txt', 'windspeedfortaleza.txt', 'samurec.txt']

# experiment_id novo e explicito (Secao 3.2 do CLAUDE.md) -- nunca reaproveitar.
# Convencao proposta na Tarefa 5 para distinguir a familia "MLP single" da
# familia "ARIMA-MLP hibrido" (que ja usa chamados_v4_fs_<metodo> sem "mlp"):
# chamados_v4_fs_mlp_<metodo>.
experiment_id = 'chamados_v4_fs_mlp_mutualinfo'
# Caminhos derivados de experiment_id, computados uma vez aqui e reusados nas
# celulas 3/4/5 (mesmo padrao ja usado pelos notebooks de ARIMA-MLP).
experiment_dir = Path(config.MODEL_DATA_PATH) / experiment_id
experiment_dir_results = Path(config.ROOT_PATH) / 'results' / experiment_id

# Sufixo sem underscore -- extract_series_name_from_path/model_name em
# metrics.py precisam disso (mesma regra ja usada por amv1<metodo> no
# hibrido). Baseline MLP single usa model_name='mlp' (sem prefixo de
# horizon, adicionado automaticamente por GridSearch/generics.format_names)
# -- aqui 'mlpmutualinfo' vira '1mlpmutualinfo' no nome do .pkl,
# mesma logica de 'amv1rfembedded' -> '1amv1rfembedded' no hibrido.
model_name = 'mlpmutualinfo'
normalize = True
force = False
model_exec = 10

experiment_params = {
    'diff_kpss': False,
    'horizon': 1,
    'type_filter': None,
}

# Convencao selector__*/estimator__* (nativa do sklearn): GridSearch usa
# clone(model).set_params(**params), que resolve essas chaves automaticamente
# via Pipeline.get_params(deep=True) -- nenhuma mudanca em grid_search_exp.py.
model_parameters = {
    'selector__k': [1, 5, 9, 15, 20],
    'estimator__hidden_layer_sizes': [10, 20, 50],
    'estimator__max_iter': [1000],
   }

In [3]:
# Sanity-check exigido pela Tarefa 2/5: confirma que Pipeline.get_params(deep=True)
# expoe as chaves selector__*/estimator__* que GridSearch (ParameterGrid +
# clone().set_params()) vai usar, antes de rodar o grid completo.
params = model.get_params(deep=True)
required_keys = {'selector__strategy', 'estimator__hidden_layer_sizes', 'estimator__max_iter'}
missing = required_keys - params.keys()
assert not missing, f"get_params(deep=True) nao expos chaves esperadas: {missing}"
print("OK -- Pipeline.get_params(deep=True) expoe:", sorted(required_keys))

# Checagem de identidade (mesmo padrao ja usado pelos notebooks de ARIMA-MLP,
# achado de code-review da Tarefa 3.2): trava que a 'strategy' do seletor
# (celula 1) e o experiment_id/model_name declarados na MESMA celula
# continuam consistentes entre si.
strategy_slug = model.named_steps['selector'].strategy.replace('_', '')
assert strategy_slug in experiment_id, (
    f"strategy={model.named_steps['selector'].strategy!r} nao aparece em "
    f"experiment_id={experiment_id!r} -- confirme que voce nao mudou 'strategy' "
    "na celula de configuracao sem atualizar experiment_id/model_name."
)
assert strategy_slug in model_name, (
    f"strategy={model.named_steps['selector'].strategy!r} nao aparece em "
    f"model_name={model_name!r} -- confirme consistencia."
)
print(f"OK -- strategy={model.named_steps['selector'].strategy!r} consistente com experiment_id={experiment_id!r} e model_name={model_name!r}")

OK -- Pipeline.get_params(deep=True) expoe: ['estimator__hidden_layer_sizes', 'estimator__max_iter', 'selector__strategy']
OK -- strategy='mutual_info' consistente com experiment_id='chamados_v4_fs_mlp_mutualinfo' e model_name='mlpmutualinfo'


In [4]:
# Chamado direto via GridSearch(...).execution() por serie, em vez de
# grid_seach_multiple_bases(), para nao depender/mutar config.BASE_NAME_LIST
# (mesmo motivo ja documentado nos notebooks de ARIMA-MLP). Diferente do
# hibrido, nao ha celula de "copia do ARIMA" -- SKlearnModel nao depende de
# nenhum modelo linear pre-treinado, opera direto sobre a serie bruta.
# use_val_slipt_for_prev=True e explicito porque o default de GridSearch
# (False) diverge do default do wrapper grid_seach_multiple_bases (True) --
# sem isso, o refit final perderia o val_size e os .pkl ficariam sem
# val_metrics, quebrando a paridade com o baseline original (mlp em
# chamados/).
for base_name in fs_series_list:
    print(base_name)
    exec_gs = grid_search_exp.GridSearch(
        single_ml_model_exp.SKlearnModel,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
    )
    exec_gs.execution()

airlines.txt
{'estimator__hidden_layer_sizes': 50, 'estimator__max_iter': 1000, 'selector__k': 5}
austres.txt
{'estimator__hidden_layer_sizes': 50, 'estimator__max_iter': 1000, 'selector__k': 15}
coloradoRiver.txt
{'estimator__hidden_layer_sizes': 50, 'estimator__max_iter': 1000, 'selector__k': 1}
sunspot.txt
{'estimator__hidden_layer_sizes': 50, 'estimator__max_iter': 1000, 'selector__k': 9}
windspeedfortaleza.txt
{'estimator__hidden_layer_sizes': 50, 'estimator__max_iter': 1000, 'selector__k': 5}
samurec.txt
{'estimator__hidden_layer_sizes': 20, 'estimator__max_iter': 1000, 'selector__k': 5}


In [5]:
# Gera o CSV de metricas (agregado + detalhado) direto no notebook -- mesma
# funcao usada pela CLI (src/utils/export_metrics_to_csv.py), so o ponto de
# entrada muda (mesmo padrao notebook-only ja usado pelo hibrido).
from utils.export_metrics_to_csv import run_export_metrics_to_csv

metrics_output = experiment_dir_results / 'metrics.csv'
df_metrics = run_export_metrics_to_csv(
    experiment_dir,
    metrics_output,
    detail=True,
)
df_metrics

[INFO] 6 arquivo(s) .pkl encontrado(s) em 'C:\Projetos\mestrado_codigos\experiments\data\result\chamados_v4_fs_mlp_mutualinfo'.

  OK  airlines_1mlpmutualinfo.pkl  ->  10 linha(s)
  OK  austres_1mlpmutualinfo.pkl  ->  10 linha(s)
  OK  coloradoRiver_1mlpmutualinfo.pkl  ->  10 linha(s)
  OK  samurec_1mlpmutualinfo.pkl  ->  10 linha(s)
  OK  sunspot_1mlpmutualinfo.pkl  ->  10 linha(s)
  OK  windspeedfortaleza_1mlpmutualinfo.pkl  ->  10 linha(s)

[OK] CSV agregado (média das repetições) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_v4_fs_mlp_mutualinfo\metrics.csv
     6 linha(s) × 20 coluna(s)

[OK] CSV detalhado (por repetição) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_v4_fs_mlp_mutualinfo\metrics_detail.csv
     60 linha(s) × 13 coluna(s)


,ExperimentID,Serie,Modelo,N_Repeticoes,MSE_mean,MSE_std,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,theil_mean,theil_std,ARV_mean,ARV_std,IA_mean,IA_std,POCID_mean,POCID_std
0,chamados_v4_fs_mlp_mutualinfo,airlines,1mlpmutualinfo,10,551.392175,171.380963,23.246105,3.497741,19.604836,2.758679,4.262349,0.542960,0.326397,0.177384,0.107721,0.044559,0.974259,0.009517,72.857143,5.634362
1,chamados_v4_fs_mlp_mutualinfo,austres,1mlpmutualinfo,10,847.870535,438.126927,28.247436,7.450047,26.181870,8.018630,0.149513,0.045767,0.443108,0.236368,0.079417,0.038819,0.978822,0.010838,87.500000,0.000000
2,chamados_v4_fs_mlp_mutualinfo,coloradoRiver,1mlpmutualinfo,10,0.055344,0.008300,0.234503,0.019804,0.196135,0.017617,26.815330,2.478471,3.704950,0.651905,1.275540,0.078170,0.638468,0.040978,51.081081,7.692027
3,chamados_v4_fs_mlp_mutualinfo,samurec,1mlpmutualinfo,10,50.127198,1.043425,7.079709,0.073938,5.735584,0.065673,19.420838,0.248249,35.733521,17.343321,20.879603,3.437272,0.251213,0.031973,51.016949,7.210864
4,chamados_v4_fs_mlp_mutualinfo,sunspot,1mlpmutualinfo,10,297.098021,20.225032,17.227444,0.589899,13.887082,0.394606,34.970953,1.341460,0.337699,0.006688,0.170830,0.010843,0.959060,0.002787,73.571429,4.517540
5,chamados_v4_fs_mlp_mutualinfo,windspeedfortaleza,1mlpmutualinfo,10,0.185823,0.000942,0.431070,0.001093,0.376444,0.001496,13.345887,0.051817,1.025793,0.037149,0.425448,0.006887,0.899187,0.000724,57.142857,0.000000


In [6]:
# Gera o CSV de features selecionadas (agregado + detalhado) direto no
# notebook -- mesma funcao usada pela CLI (src/utils/export_selected_features.py),
# so o ponto de entrada muda (mesmo padrao notebook-only ja usado pelo hibrido).
from utils.export_selected_features import run_export_selected_features

features_output = experiment_dir_results / 'selected_features.csv'
df_features = run_export_selected_features(
    experiment_dir,
    features_output,
    detail=True,
)
df_features

Failed to read module file 'C:\Projetos\mestrado_codigos\experiments\src\utils\export_metrics_to_csv.py' for module 'utils.export_metrics_to_csv': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 219, in update_sources
    self.source_by_modname[new_modname] = f.read()
                                          ^^^^^^^^
  File "C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 3950: character maps to <undefined>


[INFO] 6 arquivo(s) .pkl encontrado(s) em 'C:\Projetos\mestrado_codigos\experiments\data\result\chamados_v4_fs_mlp_mutualinfo'.

  OK  airlines_1mlpmutualinfo.pkl  ->  10 linha(s)
  OK  austres_1mlpmutualinfo.pkl  ->  10 linha(s)
  OK  coloradoRiver_1mlpmutualinfo.pkl  ->  10 linha(s)
  OK  samurec_1mlpmutualinfo.pkl  ->  10 linha(s)
  OK  sunspot_1mlpmutualinfo.pkl  ->  10 linha(s)
  OK  windspeedfortaleza_1mlpmutualinfo.pkl  ->  10 linha(s)

[OK] CSV agregado (média/desvio por série × modelo) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_v4_fs_mlp_mutualinfo\selected_features.csv
     6 linha(s) × 8 coluna(s)

[OK] CSV detalhado (por repetição) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_v4_fs_mlp_mutualinfo\selected_features_detail.csv
     60 linha(s) × 9 coluna(s)


,ExperimentID,Serie,Modelo,Strategy,N_Features_Selected_mean,N_Features_Selected_std,N_Repeticoes,N_Features_Total
0,chamados_v4_fs_mlp_mutualinfo,airlines,1mlpmutualinfo,mutual_info,5.0,0.0,10,20
1,chamados_v4_fs_mlp_mutualinfo,austres,1mlpmutualinfo,mutual_info,1.0,0.0,10,1
2,chamados_v4_fs_mlp_mutualinfo,coloradoRiver,1mlpmutualinfo,mutual_info,1.0,0.0,10,16
3,chamados_v4_fs_mlp_mutualinfo,samurec,1mlpmutualinfo,mutual_info,5.0,0.0,10,15
4,chamados_v4_fs_mlp_mutualinfo,sunspot,1mlpmutualinfo,mutual_info,9.0,0.0,10,9
5,chamados_v4_fs_mlp_mutualinfo,windspeedfortaleza,1mlpmutualinfo,mutual_info,5.0,0.0,10,20
